# STEP 6c - second exposed encoder (fp32) + the residual-visible cure

Two gaps from 6b:

1. **mDeBERTa-v3 crashed.** Its tokenizer worked, training did not. mDeBERTa-v3 is
   unstable under fp16 autocast. This notebook trains the supplementary models in
   **fp32** and tries three candidates, keeping the first two that succeed.
   Best candidate is **BanglaT5**: same lab as BanglaBERT, same language, same data
   lineage, but SentencePiece instead of WordPiece. If it is exposed and BanglaBERT
   is immune, the tokenizer is isolated as the cause.
2. **Visible attacks are not cured.** Render-equivalence removes only imperceptible
   insertions, by design, so 8.3% of the old visible attack still evaded Qwen.
   Here we add a **residual flag**: after render-equivalence, any zero-width
   character still sitting in Bangla context and not next to hasant is a visible
   manipulation. Route it to human review. We measure the catch rate and the
   false-flag rate on real comments.

Supplementary models train on a 20k subsample in fp32 (stated as such in the paper).

**Settings:** GPU **T4 x2**, Internet **ON**, data **`naurosromim/bdshs`**.
Save Version -> Save & Run All. About 60 min.


In [ ]:
!pip install -q "emoji==1.7.0" regex ftfy uharfbuzz sentencepiece protobuf
!pip install -q git+https://github.com/csebuetnlp/normalizer

In [ ]:
%%writefile /kaggle/working/render_attack.py
"""
Rendering-verified zero-width attacks and defenses for Bangla.

Threat model
------------
An insertion counts as imperceptible only if HarfBuzz shapes the word to the
same glyph ids and glyph positions, in every reference font, with and without
it. HarfBuzz is the shaper used by Chrome, Android and Firefox.

The earlier rule-based attacks (strict / semi) are kept for comparison, but
every number in the paper should come from the render-gated functions here.
"""
import os
import re
import random
import urllib.request
import unicodedata
from functools import lru_cache

import uharfbuzz as hb

ZWNJ, ZWJ, ZWSP = "‌", "‍", "​"
HASANT = "্"
ZW = (ZWNJ, ZWJ, ZWSP)

# Open-licensed Bangla fonts, all downloadable on Kaggle.
FONT_URLS = {
    "NotoSansBengali":  "https://github.com/notofonts/notofonts.github.io/raw/main/fonts/NotoSansBengali/unhinted/ttf/NotoSansBengali-Regular.ttf",
    "NotoSerifBengali": "https://github.com/notofonts/notofonts.github.io/raw/main/fonts/NotoSerifBengali/unhinted/ttf/NotoSerifBengali-Regular.ttf",
    "HindSiliguri":     "https://raw.githubusercontent.com/google/fonts/main/ofl/hindsiliguri/HindSiliguri-Regular.ttf",
    "TiroBangla":       "https://raw.githubusercontent.com/google/fonts/main/ofl/tirobangla/TiroBangla-Regular.ttf",
    "BalooDa2":         "https://raw.githubusercontent.com/google/fonts/main/ofl/balooda2/BalooDa2%5Bwght%5D.ttf",
    "AnekBangla":       "https://raw.githubusercontent.com/google/fonts/main/ofl/anekbangla/AnekBangla%5Bwdth,wght%5D.ttf",
    "Galada":           "https://raw.githubusercontent.com/google/fonts/main/ofl/galada/Galada-Regular.ttf",
    "Mina":             "https://raw.githubusercontent.com/google/fonts/main/ofl/mina/Mina-Regular.ttf",
}


def fetch_fonts(dest, names=None):
    os.makedirs(dest, exist_ok=True)
    paths = {}
    for name in names or FONT_URLS:
        p = os.path.join(dest, name + ".ttf")
        if not os.path.exists(p):
            urllib.request.urlretrieve(FONT_URLS[name], p)
        paths[name] = p
    return paths


def is_bangla(ch):
    """True for one Bangla code point.

    An empty string returns False, so callers may pass a missing neighbour
    (start or end of string) without guarding first."""
    return len(ch) == 1 and 0x0980 <= ord(ch) <= 0x09FF


# ------------------------------------------------------------------ slots

def inert_slots(word):
    """Between two Bangla chars, neither of them hasant (the old `semi` set)."""
    return [i for i in range(1, len(word))
            if is_bangla(word[i - 1]) and is_bangla(word[i])
            and HASANT not in (word[i - 1], word[i])]


def strict_slots(word):
    """Both neighbours Bangla base letters (the old `strict` set)."""
    return [i for i in inert_slots(word)
            if unicodedata.category(word[i - 1]) == "Lo"
            and unicodedata.category(word[i]) == "Lo"]


def hasant_slots(word):
    """Immediately before or after a hasant. Position-aware defense keeps these."""
    out = set()
    for h, c in enumerate(word):
        if c != HASANT:
            continue
        if h > 0 and is_bangla(word[h - 1]):
            out.add(h)
        if h + 1 < len(word) and is_bangla(word[h + 1]):
            out.add(h + 1)
    return sorted(out)


SLOT_MODES = {
    "inert":  inert_slots,
    "strict": strict_slots,
    "hasant": hasant_slots,
    "both":   lambda w: inert_slots(w) + hasant_slots(w),
}


def _build(word, chosen, char):
    return "".join((char if i in chosen else "") + c for i, c in enumerate(word))


def _parts(text):
    return re.split(r"(\s+)", text)


# --------------------------------------------------------------- renderer

class Renderer:
    def __init__(self, font_paths):
        self.fonts = {n: hb.Font(hb.Face(hb.Blob.from_file_path(p), 0))
                      for n, p in font_paths.items()}
        self.names = list(self.fonts)
        self._shape = lru_cache(maxsize=2_000_000)(self._shape_uncached)

    def _shape_uncached(self, name, text):
        buf = hb.Buffer()
        buf.add_str(text)
        buf.guess_segment_properties()
        buf.flags = hb.BufferFlags.REMOVE_DEFAULT_IGNORABLES
        hb.shape(self.fonts[name], buf, {})
        return (tuple(g.codepoint for g in buf.glyph_infos),
                tuple((p.x_advance, p.y_advance, p.x_offset, p.y_offset)
                      for p in buf.glyph_positions))

    def same(self, a, b, fonts=None):
        return all(self._shape(n, a) == self._shape(n, b) for n in (fonts or self.names))

    def visible_in(self, a, b):
        """Names of fonts in which a and b render differently."""
        return [n for n in self.names if self._shape(n, a) != self._shape(n, b)]

    # ------------------------------------------------------------ attacks

    def attack(self, text, budget="ALL", char=ZWNJ, seed=0, mode="inert", fonts=None):
        """Insert zero-width chars only where the word still renders identically.

        Candidates are shuffled and accepted one at a time, each checked against
        the clean word with all previously accepted insertions in place.
        Returns (attacked_text, n_inserted)."""
        parts = _parts(text)
        rng = random.Random(seed)
        slots = SLOT_MODES[mode]
        cands = [(pi, i) for pi, p in enumerate(parts)
                 if p and not p.isspace() for i in slots(p)]
        rng.shuffle(cands)
        chosen, n = {}, 0
        for pi, i in cands:
            if budget != "ALL" and n >= budget:
                break
            trial = chosen.get(pi, set()) | {i}
            if self.same(parts[pi], _build(parts[pi], trial, char), fonts):
                chosen[pi] = trial
                n += 1
        out = "".join(_build(p, chosen[pi], char) if pi in chosen else p
                      for pi, p in enumerate(parts))
        return out, n

    # ------------------------------------------------------------ defense

    def defend_render_equivalence(self, text, fonts=None):
        """Delete a zero-width char iff removing it leaves rendering unchanged in
        every reference font. Only chars whose nearest non-zero-width neighbours
        on both sides are Bangla are considered, so emoji ZWJ sequences survive."""
        parts = _parts(text)
        for pi, w in enumerate(parts):
            if not any(z in w for z in ZW):
                continue
            i = 0
            while i < len(w):
                if w[i] in ZW:
                    l = i - 1
                    while l >= 0 and w[l] in ZW:
                        l -= 1
                    r = i + 1
                    while r < len(w) and w[r] in ZW:
                        r += 1
                    if l >= 0 and r < len(w) and is_bangla(w[l]) and is_bangla(w[r]):
                        cand = w[:i] + w[i + 1:]
                        if self.same(w, cand, fonts):
                            w = cand
                            continue
                i += 1
            parts[pi] = w
        return "".join(parts)


# ------------------------------------------------ rule-based (for comparison)

def attack_ungated(text, budget="ALL", char=ZWNJ, seed=0, mode="inert"):
    parts = _parts(text)
    rng = random.Random(seed)
    slots = SLOT_MODES[mode]
    cands = [(pi, i) for pi, p in enumerate(parts) if p and not p.isspace() for i in slots(p)]
    rng.shuffle(cands)
    if budget != "ALL":
        cands = cands[:budget]
    chosen = {}
    for pi, i in cands:
        chosen.setdefault(pi, set()).add(i)
    out = "".join(_build(p, chosen[pi], char) if pi in chosen else p for pi, p in enumerate(parts))
    return out, len(cands)


def defend_strip_all(text):
    for z in ZW:
        text = text.replace(z, "")
    return text


def defend_nfc(text):
    return unicodedata.normalize("NFC", text)


def defend_position_aware(text):
    out = []
    for i, ch in enumerate(text):
        if ch in (ZWNJ, ZWJ):
            prev = text[i - 1] if i > 0 else ""
            nxt = text[i + 1] if i + 1 < len(text) else ""
            if HASANT in (prev, nxt):
                out.append(ch)
            continue
        if ch == ZWSP:
            continue
        out.append(ch)
    return "".join(out)


def zw_stats(texts):
    docs = total = functional = 0
    for t in texts:
        hit = False
        for i, ch in enumerate(t):
            if ch not in (ZWNJ, ZWJ):
                continue
            hit = True
            total += 1
            prev = t[i - 1] if i > 0 else ""
            nxt = t[i + 1] if i + 1 < len(t) else ""
            functional += HASANT in (prev, nxt)
        docs += hit
    return dict(n_docs=len(texts), docs_with_zw=docs, total_zw=total,
                functional=functional, inert=total - functional)


In [ ]:
import sys, os, glob, time, random, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "/kaggle/working")
import importlib, render_attack as ra; importlib.reload(ra)
import numpy as np, pandas as pd, torch
from functools import lru_cache
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, accuracy_score
from normalizer import normalize

DEV, OUT = "cuda", "/kaggle/working"
R = ra.Renderer(ra.fetch_fonts(f"{OUT}/fonts"))
paths = glob.glob("/kaggle/input/**/*.csv", recursive=True)
pick = lambda k: [p for p in paths if k in os.path.basename(p).lower()][0]
rtr, rte = pd.read_csv(pick("train")), pd.read_csv(pick("test"))
obj = [c for c in rtr.columns if rtr[c].dtype == object]
TEXT = max(obj, key=lambda c: rtr[c].astype(str).str.len().mean())
LABEL = [c for c in rtr.columns if c != TEXT and rtr[c].dropna().nunique() == 2][0]
def prep(d):
    d = d[[TEXT, LABEL]].dropna().copy(); d.columns = ["text", "label"]
    d["text"] = d["text"].astype(str); d["label"] = d["label"].astype(int)
    return d[d.text.str.strip().str.len() > 0].reset_index(drop=True)
train_full, test = prep(rtr), prep(rte)
train = train_full.sample(20000, random_state=42).reset_index(drop=True)
HATE = 1

N_ATK = 1500
rng = random.Random(0)
hate_txt = [test.text[i] for i in sorted(rng.sample(test.index[test.label == HATE].tolist(), N_ATK))]
ben_txt  = [test.text[i] for i in sorted(rng.sample(test.index[test.label != HATE].tolist(), N_ATK))]
ATK = {
    "gated_inert_ALL":      [R.attack(t, "ALL", mode="inert", seed=i)[0] for i, t in enumerate(hate_txt)],
    "gated_both_ALL":       [R.attack(t, "ALL", mode="both", seed=i)[0] for i, t in enumerate(hate_txt)],
    "ungated_semi_ALL_old": [ra.attack_ungated(t, "ALL", mode="inert", seed=i)[0] for i, t in enumerate(hate_txt)],
}
print("train", len(train), "| test", len(test), "| attacked", len(hate_txt))

In [ ]:
# PART 1 - the residual-visible cure (CPU only, no model needed)
def residual_flag(text):
    """Render-equivalence first, then flag what survives.

    After render-equivalence every remaining zero-width character is either
    functional (hasant-adjacent, normal Bangla) or a manipulation a reader can
    see. The second kind is flagged for human review."""
    t = R.defend_render_equivalence(text)
    n = 0
    for i, ch in enumerate(t):
        if ch not in ra.ZW:
            continue
        prev = t[i - 1] if i else ""
        nxt = t[i + 1] if i + 1 < len(t) else ""
        if ra.HASANT in (prev, nxt):
            continue
        if ra.is_bangla(prev) or ra.is_bangla(nxt):
            n += 1
    return t, n

rows = []
for name, texts in ATK.items():
    flagged = np.mean([residual_flag(t)[1] > 0 for t in texts])
    rows.append({"set": name, "kind": "attacked", "n": len(texts), "flag_rate": round(float(flagged), 4)})
alltxt = train_full.text.tolist() + test.text.tolist()
zw_docs = [t for t in alltxt if any(z in t for z in ra.ZW)]
rows.append({"set": "real comments containing zero-width", "kind": "benign", "n": len(zw_docs),
             "flag_rate": round(float(np.mean([residual_flag(t)[1] > 0 for t in zw_docs])), 4)})
rows.append({"set": "all clean test comments", "kind": "benign", "n": len(test),
             "flag_rate": round(float(np.mean([residual_flag(t)[1] > 0 for t in test.text])), 4)})
flag = pd.DataFrame(rows)
flag.to_csv(f"{OUT}/step6c_residual_flag.csv", index=False)
print(flag.to_string(index=False))

In [ ]:
# PART 2 - supplementary encoders, fp32 (no autocast: mDeBERTa-v3 NaNs under fp16)
MAXLEN = 128
@lru_cache(maxsize=None)
def norm(t): return normalize(t)

DEFENSES = {"none": lambda t: t, "nfc": ra.defend_nfc, "strip_all": ra.defend_strip_all,
            "position_aware": ra.defend_position_aware, "render_equivalence": R.defend_render_equivalence}
DEFENDED = {(a, d): [fn(t) for t in ATK[a]] for a in ATK for d, fn in DEFENSES.items()}
CLEAN_DEF = {d: [fn(t) for t in test.text] for d, fn in DEFENSES.items()}

def run(name, family, seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    tk = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2).to(DEV)
    if mdl.config.pad_token_id is None:
        mdl.config.pad_token_id = tk.pad_token_id
    e = tk([norm(t) for t in train.text], truncation=True, padding="max_length",
           max_length=MAXLEN, return_tensors="pt")
    y = torch.tensor(train.label.values)
    g = torch.Generator(); g.manual_seed(seed)
    dl = DataLoader(TensorDataset(e["input_ids"], e["attention_mask"], y), batch_size=16, shuffle=True, generator=g)
    opt = torch.optim.AdamW(mdl.parameters(), lr=2e-5)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-5, total_steps=2 * len(dl), pct_start=0.1)
    mdl.train()
    for ep in range(2):
        for st, (i_, m_, l_) in enumerate(dl):
            i_, m_, l_ = i_.to(DEV), m_.to(DEV), l_.to(DEV)
            opt.zero_grad()
            loss = mdl(input_ids=i_, attention_mask=m_, labels=l_).loss   # fp32
            if not torch.isfinite(loss):
                raise RuntimeError("non-finite loss")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            opt.step(); sch.step()
            if st % 400 == 0: print(f"  {name} ep{ep} {st}/{len(dl)} {loss.item():.4f}")
    mdl.eval()

    @torch.no_grad()
    def pred(texts, bs=64):
        ids = tk([norm(t) for t in texts], truncation=True, max_length=512)["input_ids"]
        order = np.argsort([len(x) for x in ids]); out = np.empty(len(texts), dtype=int)
        for s in range(0, len(order), bs):
            idx = order[s:s + bs]
            b = tk.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt")
            lg = mdl(input_ids=b["input_ids"].to(DEV), attention_mask=b["attention_mask"].to(DEV)).logits
            out[idx] = lg.float().argmax(-1).cpu().numpy()
        return out

    ty = test.label.values
    clean = [{"model": name, "family": family, "defense_on_clean": d,
              "acc": round(accuracy_score(ty, pred(CLEAN_DEF[d])), 4),
              "f1": round(f1_score(ty, pred(CLEAN_DEF[d])), 4)} for d in ["none"]]
    hp = pred(hate_txt); tgt = np.where(hp == HATE)[0]
    lad, dfn = [], []
    for cfg in ATK:
        p = pred([ATK[cfg][i] for i in tgt])
        lad.append({"model": name, "family": family, "tokenizer": type(tk).__name__, "config": cfg,
                    "n_targets": len(tgt), "evasion": round(float((p != HATE).mean()), 4)})
        print("  ", lad[-1])
    for a in ATK:
        for d in DEFENSES:
            p = pred([DEFENDED[(a, d)][i] for i in tgt])
            dfn.append({"model": name, "attack": a, "defense": d,
                        "evasion": round(float((p != HATE).mean()), 4)})
    bp, bpa = pred(ben_txt), pred([ATK["gated_both_ALL"][i] for i in range(len(ben_txt))][:len(ben_txt)])
    del mdl; torch.cuda.empty_cache()
    return clean, lad, dfn

In [ ]:
CANDIDATES = [("csebuetnlp/banglat5", "SentencePiece"),
              ("microsoft/mdeberta-v3-base", "SentencePiece"),
              ("google/mt5-base", "SentencePiece")]
CL, LA, DE, STATUS = [], [], [], []
for name, fam in CANDIDATES:
    if len([s for s in STATUS if s["ok"]]) >= 2:
        STATUS.append({"model": name, "ok": False, "error": "skipped, 2 already succeeded}"[:-1]}); continue
    t0 = time.time()
    try:
        c, l, d = run(name, fam)
        CL += c; LA += l; DE += d
        STATUS.append({"model": name, "ok": True, "error": ""})
        print(f"== {name} OK in {(time.time()-t0)/60:.1f} min")
    except Exception as ex:
        STATUS.append({"model": name, "ok": False, "error": f"{type(ex).__name__}: {str(ex)[:120]}"})
        print(f"== {name} FAILED: {type(ex).__name__}: {ex}")
        torch.cuda.empty_cache()
    pd.DataFrame(CL).to_csv(f"{OUT}/step6c_clean.csv", index=False)
    pd.DataFrame(LA).to_csv(f"{OUT}/step6c_ladder.csv", index=False)
    pd.DataFrame(DE).to_csv(f"{OUT}/step6c_defense.csv", index=False)
    pd.DataFrame(STATUS).to_csv(f"{OUT}/step6c_status.csv", index=False)

In [ ]:
pd.set_option("display.width", 220)
print("=" * 80); print("STEP 6c SUMMARY"); print("=" * 80)
print("-- residual-visible cure --"); print(flag.to_string(index=False))
print("\n-- model status --");  print(pd.DataFrame(STATUS).to_string(index=False))
print("\n-- evasion --");       print(pd.DataFrame(LA).to_string(index=False))
print("\n-- defense --");       print(pd.DataFrame(DE).to_string(index=False))
print("\n-- clean --");         print(pd.DataFrame(CL).to_string(index=False))
print("=" * 80)